# Experimenting with effect of missing uv-coverage or noise in SA data
## A. Ordog, Feb 13, 2023
### Feb 14: added simple gap test
### Feb 21: option to test different feathering and optimize
### Feb 22: option to run multiple simulations and optimize feathering for each

In [1]:
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib import pylab
from PIL import Image
from math import e
import astropy.io.fits as pf
from astropy.io import fits
from mpl_toolkits.axes_grid1 import make_axes_locatable
from tqdm import tqdm

## Read in files (change directory names accordingly)

In [2]:
#######################################################
dir_in = '/home/aordog/DATA/CGPS_FITS_files/' # on faraday
#dir_in = '/home2/DATA_AO/CGPS_FITS_files/'    # on DRAO desktop
#######################################################

CGPS = fits.open(dir_in+'CGPS_C21.fits')
hdr = CGPS[0].header
data_full = CGPS[0].data[0,0]
print(data_full.shape)
print(hdr['CDELT1'])
#print(repr(hdr))

CGPS2 = fits.open(dir_in+'all_stokesi_allbands9.fits')
hdr2 = CGPS2[0].header
data_ST = CGPS2[0].data#[0,0]
print(data_ST.shape)
print(hdr2['CDELT1'])

FileNotFoundError: [Errno 2] No such file or directory: '/home/aordog/DATA/CGPS_FITS_files/CGPS_C21.fits'

## Define functions

In [ ]:
def gf(mu,fwhm,x):
    return np.exp(-4*np.log(2)*((x-mu)**2)/fwhm**2)

In [ ]:
def regular_image(data,j0,i0,nxy,taper=False,taper_width=50,*args,**kwargs):

    image = data[j0:j0+nxy,i0:i0+nxy]

    dxy = np.round(hdr['CDELT2'],5)
    pix0 = int((nxy-1)/2)
    widxy = nxy*dxy
    
    if taper:      
        x, y = np.meshgrid(np.linspace(-(nxy-1)/2,(nxy-1)/2,nxy), 
                       np.linspace(-(nxy-1)/2,(nxy-1)/2,nxy))
        r = np.sqrt(x**2+y**2)
        
        taper = gf(nxy/2-taper_width,taper_width,r)
        taper[np.where(r<=nxy/2-taper_width)] = 1
        image = image*taper

    print('Width of each pixel: '+str(dxy)+ ' deg.')
    print('Number of x and y pixels: '+str(nxy))
    print('Central pixel index: '+str(pix0))
    print('Width of image: '+str(widxy)+' deg.')
    
    return image,dxy,pix0,widxy


In [ ]:
def padded_image(data,j0,i0,nxy,taper=False,taper_width=50,*args,**kwargs):

    image_small = data[j0:j0+nxy,i0:i0+nxy]

    dxy = np.round(hdr['CDELT2'],5)
    pix0 = int((nxy-1)/2)+int((nxy+1)/2)
    widxy = nxy*dxy
    
    if taper:      
        x, y = np.meshgrid(np.linspace(-(nxy-1)/2,(nxy-1)/2,nxy), 
                       np.linspace(-(nxy-1)/2,(nxy-1)/2,nxy))
        r = np.sqrt(x**2+y**2)
        
        taper = gf(nxy/2-taper_width,taper_width,r)
        taper[np.where(r<=nxy/2-taper_width)] = 1
        image_small = image_small*taper
        plt.plot(taper[511,:])
        
    image = np.zeros([2*nxy+1,2*nxy+1])
    image[:,:] = np.nanmean(image_small)
    image[int((nxy+1)/2):int((nxy+1)/2)+nxy,int((nxy+1)/2):int((nxy+1)/2)+nxy] = image_small

    print('Width of each pixel: '+str(dxy)+ ' deg.')
    print('Number of x and y pixels: '+str(nxy))
    print('Central pixel index: '+str(pix0))
    print('Width of image: '+str(widxy)+' deg.')
    
    return image,dxy,pix0,widxy

In [ ]:
def get_beam(image,dxy,pix0,R=9,plots=True,*arg,**kwargs):
    
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)
    
    extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    mask = gf(0,2*R,ruv)
    
    if plots:
        # Plot original image:
        fig,ax = plt.subplots(2,2,figsize=(12,10))
        ax[1,0].set_title('Original image')
        ax[1,0].imshow(image,origin='lower',vmin=0,vmax=30)
        ax[0,0].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
    
    # Apply beam mask and plot:
    image_FFT_shift = image_FFT_shift*mask
    image_SA_nonoise = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
    
    if plots:
        ax[1,1].set_title('Beam mask applied')
        ax[1,1].imshow(image_SA_nonoise.real,origin='lower',vmin=0,vmax=30)
        ax[0,1].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)   
    
    return mask, ruv[pix0,:],mask[pix0,:],image_SA_nonoise.real


In [ ]:
### NOTE: OBSOLETE!!!
def SA_observe(image,dxy,pix0,noise_slope=0,R=9,Rnoise=9,*arg,**kwargs):
    
    # Note: default is no noise added (noise_slope = 0)
    
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)
    
    extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    mask = gf(0,2*R,ruv)
    noise = np.random.normal(size=([image.shape[0],image.shape[1]]))
    noise = noise*np.sqrt(ruv-Rnoise)*noise_slope
    noise[ruv<Rnoise] = 0

    fig,ax = plt.subplots(2,2,figsize=(12,10))
    ax[1,0].set_title('Beam mask')
    ax[0,0].imshow(mask,origin='lower',vmin=0,vmax=1,extent=extent)
    ax[1,0].plot(uv_m,mask[pix0,:]), ax[1,0].set_box_aspect(1)
    ax[1,1].set_title('Added noise')
    ax[0,1].imshow(noise,origin='lower',vmin=-200,vmax=200,extent=extent)
    ax[1,1].plot(uv_m,noise[pix0,:]), ax[1,1].set_box_aspect(1)
    
    # Plot original image:
    fig,ax = plt.subplots(2,3,figsize=(16,10))
    ax[1,0].set_title('Original image')
    ax[1,0].imshow(image,origin='lower',vmin=0,vmax=30)
    ax[0,0].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
    
    # Apply beam mask and plot:
    image_FFT_shift = image_FFT_shift*mask
    image_SA_nonoise = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
    ax[1,1].set_title('Beam mask applied')
    ax[1,1].imshow(image_SA_nonoise.real,origin='lower',vmin=0,vmax=30)
    ax[0,1].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
        
    # Apply noise and plot:    
    image_FFT_shift = image_FFT_shift+noise
    image_SA = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
    ax[1,2].set_title('Simulated SA obs: beam + noise applied')
    ax[1,2].imshow(image_SA.real,origin='lower',vmin=0,vmax=30)
    ax[0,2].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)

    #mask = np.zeros_like(image)
    #npix1 = 2*int(np.round(R*np.diff(uv_m)[0],0))
    #npix = npix1*2+1
    #window1d = np.hanning(npix)
    #window2d = np.outer(window1d,window1d)
    #mask[pix0-npix1:pix0+npix1+1,pix0-npix1:pix0+npix1+1] = window2d    
    
    return image_SA, mask, noise

In [ ]:
### NOTE: OBSOLETE!!!
def SA_deconvolve(image, image_full, SA_beam):
    
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)
    
    image_full_FFT = np.fft.fft2(image_full)
    image_full_FFT_shift = np.fft.fftshift(image_full_FFT)

    uv_freq = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)
    
    extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    # Plot simulated observation (SA image):
    fig,ax = plt.subplots(2,2,figsize=(12,10))
    ax[1,0].set_title('Simulated SA observation')
    ax[1,0].imshow(SA.real,origin='lower',vmin=0,vmax=30)
    ax[0,0].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
    
    image_FFT_shift[SA_beam>1e-300] = image_FFT_shift[SA_beam>1e-300]/SA_beam[SA_beam>1e-300]
    image_FFT_shift[SA_beam<1e-300] = 1e300
    print('Threshold for zero: ',np.min(ruv[SA_beam<1e-300]))
    #SA_deconv = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
            
    # Plot deconvolved:
    ax[1,1].set_title('SA beam deconvolved')
    ax[0,1].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
    ax[1,1].plot(uv_m,np.log10(abs(image_FFT_shift[511,:])))
    ax[1,1].plot(uv_m,np.log10(abs(image_full_FFT_shift[511,:])))
    ax[1,1].set_ylim(2,8),ax[1,1].set_xlim(-40,40),ax[1,1].grid()
     
    return image_FFT_shift


In [ ]:
def SA_deconvolve2(image_full, SA_beam, noiseslope, plots=True, *args,**kwargs):
    
    image_full_FFT = np.fft.fft2(image_full)
    image_full_FFT_shift = np.fft.fftshift(image_full_FFT)

    uv_freq = np.fft.fftshift(np.fft.fftfreq(image_full.shape[0]))/dxy
    uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)
    
    extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    noise = noiseslope*np.random.normal(size=([image_full.shape[0],image_full.shape[1]]))
   
    if plots:
        # Plot simulated observation (SA image):
        fig,ax = plt.subplots(2,2,figsize=(12,10))
        ax[1,0].set_title('Original image')
        ax[1,0].imshow(image_full.real,origin='lower',vmin=0,vmax=30)
        ax[0,0].imshow(abs(image_full_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
    
    image_FFT_shift = np.empty_like(image_full_FFT_shift)
    image_FFT_shift[SA_beam>1e-300] = image_full_FFT_shift[SA_beam>1e-300]+noise[SA_beam>1e-300]/SA_beam[SA_beam>1e-300]
    #image_FFT_shift[SA_beam<1e-300] = 1e300
    print('Threshold for zero: ',np.min(ruv[SA_beam<1e-300]))
    
    if plots:
        # Plot deconvolved:
        ax[1,1].set_title('SA beam deconvolved')
        ax[0,1].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
        ax[1,1].plot(uv_m,np.log10(abs(image_FFT_shift[511,:])))
        ax[1,1].plot(uv_m,np.log10(abs(image_full_FFT_shift[511,:])))
        ax[1,1].set_ylim(0,10),ax[1,1].set_xlim(-40,40),ax[1,1].grid()
     
    return image_FFT_shift


In [ ]:
def ST_observe(image,image_ST_true,dxy,pix0,R1=0,R2=13,plots=True,*arg,**kwargs):
    
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)
    #print(uv_m) # Units of 1/deg

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)

    extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    b = np.pi/(R2-R1)
    r0 = (R1+R2)/2.
    mask = 0.5*np.sin(b*(ruv - r0))+0.5
    mask[ruv > R2] = 1
    mask[ruv < R1] = 0
    
    if plots:
    
        fig,ax = plt.subplots(1,1,figsize=(6,2))
        ax.plot(ruv[pix0,:],mask[pix0,:]),ax.set_xlim(0,50),ax.grid()        
        # Plot original image:
        fig,ax = plt.subplots(2,3,figsize=(16,10))
        ax[1,0].set_title('Original image')
        ax[1,0].imshow(image,origin='lower',vmin=0,vmax=30)
        ax[0,0].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
    
    # Apply ST cutoff and plot:
    image_FFT_shift = image_FFT_shift*mask
    image_ST = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
    
    if plots:    
        ax[1,1].set_title('Simulated ST')
        ax[1,1].imshow(image_ST.real,origin='lower',vmin=-20,vmax=20,cmap='RdBu_r')
        ax[0,1].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
        
    # plot actual ST:    
    image_FFT_true = np.fft.fft2(image_ST_true)
    image_FFT_true_shift = np.fft.fftshift(image_FFT_true)
    
    if plots:
        ax[1,2].set_title('Actual ST')
        ax[1,2].imshow(image_ST_true,origin='lower',vmin=-0.2,vmax=0.2,cmap='RdBu_r')
        ax[0,2].imshow(abs(image_FFT_true_shift),origin='lower',vmin=0,vmax=1e3,extent=extent)
        
    return image_ST, ruv[pix0,:],mask[pix0,:],image_ST.real


In [ ]:
def feather_ST(image,dxy,pix0,R1=12.9,R2=17.1,*arg,**kwargs):
        
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)
    #print(uv_m) # Units of 1/deg

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)

    #extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    b = np.pi/(R2-R1)
    r0 = (R1+R2)/2.
    mask = 0.5*np.sin(b*(ruv - r0))+0.5
    mask[ruv > R2] = 1
    mask[ruv < R1] = 0
    
    image_FFT_shift = image_FFT_shift*mask
    image_ST = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
  
    return image_ST, mask, uv_m

In [ ]:
def feather_SA(image,FFT,dxy,pix0,R1=12.9,R2=17.1,*arg,**kwargs):
        
    #image_FFT = np.fft.fft2(image)
    #image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)
    #print(uv_m) # Units of 1/deg

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)

    #extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    b = np.pi/(R2-R1)
    r0 = (R1+R2)/2.
    mask = -0.5*np.sin(b*(ruv - r0))+0.5
    mask[ruv > R2] = 0
    mask[ruv < R1] = 1
    
    FFT = FFT*mask
    #plt.imshow(abs(FFT),vmin=0,vmax=1e5)
    
    image_SA = np.fft.ifft2(np.fft.ifftshift(FFT))
       
  
    return image_SA, mask, uv_m

In [ ]:
def simple_gap(image,dxy,pix0,R1=14,R2=18,R3=9,R4=13,*arg,**kwargs):
        
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)
    #print(uv_m) # Units of 1/deg

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)

    #extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    b = np.pi/(R2-R1)
    r0 = (R1+R2)/2.
    mask1 = 0.5*np.sin(b*(ruv - r0))+0.5
    mask1[ruv > R2] = 1
    mask1[ruv < R1] = 0
    
    b = np.pi/(R4-R3)
    r0 = (R3+R4)/2.
    mask2 = -0.5*np.sin(b*(ruv - r0))+0.5
    mask2[ruv > R4] = 0
    mask2[ruv < R3] = 1
    
    mask = mask1+mask2
    
    image_FFT_shift = image_FFT_shift*mask
    image_new = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
  
    return image_new, mask, uv_m

In [ ]:
def feathering_tests(ST,image_full,SA_deconv_FFT,dxy,pix0,dR):

    R1_arr = np.arange(3,26,dR)
    R2_arr = np.arange(3,26,dR)

    mindiff = np.empty([len(R2_arr),len(R1_arr)])
    maxdiff = np.empty([len(R2_arr),len(R1_arr)])
    print(mindiff.shape)

    #for freq_idx in tqdm(range(0, len(freq))):
    for j in tqdm(range(0,len(R2_arr))):
        #print(j)
        for i in range(0,len(R1_arr)):
            R1 = R1_arr[i]
            R2 = R2_arr[j]
            if R2>R1:
                ST_feather,ST_mask,uv_m_ST = feather_ST(ST,dxy,pix0,R1=R1,R2=R2)
                SA_feather,SA_mask,uv_m_SA = feather_SA(image_full,SA_deconv_FFT,dxy,pix0,R1=R1,R2=R2)
                frac_diff = (abs(SA_feather+ST_feather)-image_full)/image_full
                mindiff[j,i] = np.min(frac_diff)*100
                maxdiff[j,i] = np.max(frac_diff)*100
            else:
                mindiff[j,i] = np.nan
                maxdiff[j,i] = np.nan
                
    return R1_arr, R2_arr, mindiff, maxdiff


def find_optimized_feathering(R1_arr,R2_arr,dR,mindiff,maxdiff,SA_noise,ST_R):

    w_low = np.where(maxdiff == np.nanmin(maxdiff))
    print(maxdiff[w_low][0])
    print(R1_arr[w_low[1]])
    print(R2_arr[w_low[0]])

    fs = 16
    fig,ax = plt.subplots(1,figsize=(10,10))
    extent = (R1_arr[0]-dR/2,R1_arr[-1]+dR/2,R2_arr[0]-dR/2,R2_arr[-1]+dR/2)
    im = ax.imshow(maxdiff,vmin=1,vmax=20,extent=extent,origin='lower')
    ax.set_xlabel('Maximum SA baseline, R1 (m)',fontsize = fs)
    ax.set_ylabel('Minimum ST baseline, R2 (m)',fontsize = fs)
    ax.set_title('ST edge = '+str(round(ST_R,1))+' m,   SA noise slope = '+str(round(SA_noise,1)),fontsize=fs)
    #ax.grid()

    ax.scatter(R1_arr[w_low[1]],R2_arr[w_low[0]],s=5,color='r')
    ax.text(14,10,'R1 = '+str(round(R1_arr[w_low[1][0]],1))+' m',fontsize = fs)
    ax.text(14,8,'R2 = '+str(round(R2_arr[w_low[0][0]],1))+' m',fontsize = fs)
    ax.text(14,6,'Max diff. = '+str(round(maxdiff[w_low][0],1))+' %',fontsize = fs)
    ax.tick_params(axis='both', labelsize=fs)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='5%', pad=0.05)
    cbar = fig.colorbar(im, cax=cax, orientation='vertical')
    cbar.ax.tick_params(labelsize=fs) 
    cbar.set_label('Percent difference', fontsize=fs)

    plt.tight_layout()
    plt.savefig('/home/ordoga/Python/CGPS_GMIMS_PLOTS/optimize_uv_'+str(int(ST_R))+'_'+str(int(SA_noise))+'.png')
    
    return

## Make images from full-uv-coverage data and ST-only data (for comparison)

In [ ]:
image_full,dxy,pix0,widxy = regular_image(data_full,400,22100,1023)
print('')
image_ST_true,dxy,pix0,widxy = regular_image(data_ST,400,22555,1023)

fig,ax = plt.subplots(1,2,figsize=(12,6)) 
ax[0].imshow(image_full,origin='lower',vmin=0,vmax=30), ax[0].set_title('Full ST+SA image')
ax[1].imshow(image_ST_true,origin='lower',vmin=0,vmax=0.2),ax[1].set_title('ST only (actual)')

# (1) Single simulation with plots

## Simulate ST data from full-coverage data

In [ ]:
#ST_R = 13.
ST_R = 13.
ST,r_ST,mask_ST, ST_map = ST_observe(image_full,image_ST_true,dxy,pix0,R1=4,R2=ST_R,plots=True)

## Simulate SA data from full-coverage data and get SA beam

In [ ]:
#SA, SA_beam, SA_noise = SA_observe(image_full,dxy,pix0,R=9,Rnoise=0,noise_slope=500)
SA_beam, r_SA, mask_SA, SA_map = get_beam(image_full,dxy,pix0,R=9,plots=True)

## Simulate effect of noise in beam deconvolution step

In [ ]:
#SA_deconv_FFT = SA_deconvolve(SA, image_full, SA_beam)
SA_noise = 2000
SA_deconv_FFT = SA_deconvolve2(image_full, SA_beam, SA_noise,plots=True)

## Feather simulated data in uv-plane and combine in image plane

In [ ]:
#ST_feather,ST_mask,uv_m_ST = feather_ST(ST,dxy,pix0,R1=9,R2=17)
#SA_feather,SA_mask,uv_m_SA = feather_SA(image_full,SA_deconv_FFT,dxy,pix0,R1=9,R2=17)

ST_feather,ST_mask,uv_m_ST = feather_ST(ST,dxy,pix0,R1=9,R2=13)
SA_feather,SA_mask,uv_m_SA = feather_SA(image_full,SA_deconv_FFT,dxy,pix0,R1=9,R2=13)

fig,ax = plt.subplots(1,1,figsize=(6,3))

#ax.fill_between(X, Y, 0, color='blue', alpha=.1)


ax.plot(uv_m_ST,ST_mask[pix0,:],color='red')
ax.plot(uv_m_SA,SA_mask[pix0,:],color='blue')
ax.plot(r_ST[511:1024],mask_ST[511:1024],color='k',linewidth=0.5)
ax.plot(r_SA[511:1024],mask_SA[511:1024],color='k',linewidth=0.5)
ax.axvline(x=9,color='blue',linestyle='dashed')
ax.axvline(x=13,color='red',linestyle='dashed')
#ax.plot(r_SA[511:1024],mask_SA[511:1024]+mask_ST[511:1024],color='k')
ax.fill_between(r_ST[511:1024],0,mask_ST[511:1024],color='gray',alpha=0.2,linewidth=2)
ax.fill_between(r_SA[511:1024],0,mask_SA[511:1024],color='gray',alpha=0.2,linewidth=2)
ax.set_xlim(0,30)
ax.set_ylim(0,1.1)
ax.set_xlabel('Baseline (m)')
plt.tight_layout()
plt.savefig('../CGPS_GMIMS_plots/filter_sketch.png')

#print(r_ST[511:1024])

#fig,ax = plt.subplots(1,3,figsize=(20,6))
#ax[0].imshow(SA_feather.real,origin='lower',vmin=0,vmax=30)
#ax[1].imshow(ST_feather.real,origin='lower',vmin=0,vmax=30)
#ax[2].imshow(abs(SA_feather+ST_feather),origin='lower',vmin=0,vmax=30)


In [ ]:
fig,ax = plt.subplots(1,3,figsize=(20,6))
ax[0].imshow(abs(SA_feather+ST_feather),origin='lower',vmin=0,vmax=30)
ax[1].imshow(image_full,origin='lower',vmin=0,vmax=30)

frac_diff = (abs(SA_feather+ST_feather)-image_full)/image_full

ax[2].imshow(frac_diff,origin='lower',vmin=-0.05,vmax=0.05)
print(np.min(frac_diff),np.max(frac_diff))

In [ ]:
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

In [ ]:
fig,ax = plt.subplots(1,5,figsize=(17,4))
fs = 16

im = ax[0].imshow(image_full,origin='lower',vmin=0,vmax=30,cmap='gray')
ax[0].set_title('Full-scale (true) image',fontsize=fs)
ax2_divider = make_axes_locatable(ax[0])
cax2 = ax2_divider.append_axes("bottom", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="horizontal",label='K')

im = ax[1].imshow(SA_map,origin='lower',vmin=0,vmax=30,cmap='gray')
ax[1].set_title('Single dish observation',fontsize=fs)
ax2_divider = make_axes_locatable(ax[1])
cax2 = ax2_divider.append_axes("bottom", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="horizontal",label='K')

im = ax[2].imshow(ST_map,origin='lower',vmin=0,vmax=10,cmap='gray')
ax[2].set_title('Interferometer observation',fontsize=fs)
ax2_divider = make_axes_locatable(ax[2])
cax2 = ax2_divider.append_axes("bottom", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="horizontal",label='K')

im = ax[3].imshow(abs(SA_feather+ST_feather),origin='lower',vmin=0,vmax=30,cmap='gray')
ax[3].set_title('Combined reconstruction',fontsize=fs)
ax2_divider = make_axes_locatable(ax[3])
cax2 = ax2_divider.append_axes("bottom", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="horizontal",label='K')

frac_diff = (abs(SA_feather+ST_feather)-image_full)/image_full
im = ax[4].imshow(frac_diff,origin='lower',vmin=-0.05,vmax=0.05,cmap='bwr')
ax[4].set_title('True - combined',fontsize=fs)
ax2_divider = make_axes_locatable(ax[4])
cax2 = ax2_divider.append_axes("bottom", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="horizontal",label='Fractional residuals')

print(np.min(frac_diff),np.max(frac_diff))

for i in range(0,5):
    ax[i].xaxis.set_tick_params(labelbottom=False)
    ax[i].yaxis.set_tick_params(labelleft=False)
    ax[i].set_xticks([])
    ax[i].set_yticks([])

plt.tight_layout()
plt.savefig('../CGPS_GMIMS_plots/filtering.png')

In [ ]:
fig,ax = plt.subplots(2,3,figsize=(11,6.5))
fs = 14

im = ax[0,0].imshow(image_full,origin='lower',vmin=0,vmax=30,cmap='gray')
ax[0,0].set_title('Full-scale (true) image',fontsize=fs)
ax2_divider = make_axes_locatable(ax[0,0])
cax2 = ax2_divider.append_axes("right", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="vertical",label='K')

im = ax[0,1].imshow(SA_map,origin='lower',vmin=0,vmax=30,cmap='gray')
ax[0,1].set_title('Single dish observation',fontsize=fs)
ax2_divider = make_axes_locatable(ax[0,1])
cax2 = ax2_divider.append_axes("right", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="vertical",label='K')

im = ax[0,2].imshow(ST_map,origin='lower',vmin=0,vmax=10,cmap='gray')
ax[0,2].set_title('Interferometer observation',fontsize=fs)
ax2_divider = make_axes_locatable(ax[0,2])
cax2 = ax2_divider.append_axes("right", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="vertical",label='K')


ax[1,0].plot(uv_m_ST,ST_mask[pix0,:],color='red')
ax[1,0].plot(uv_m_SA,SA_mask[pix0,:],color='blue')
ax[1,0].plot(r_ST[511:1024],mask_ST[511:1024],color='k',linewidth=0.5)
ax[1,0].plot(r_SA[511:1024],mask_SA[511:1024],color='k',linewidth=0.5)
ax[1,0].axvline(x=9,color='blue',linestyle='dashed')
ax[1,0].axvline(x=13,color='red',linestyle='dashed')
#ax.plot(r_SA[511:1024],mask_SA[511:1024]+mask_ST[511:1024],color='k')
ax[1,0].fill_between(r_ST[511:1024],0,mask_ST[511:1024],color='gray',alpha=0.2,linewidth=2)
ax[1,0].fill_between(r_SA[511:1024],0,mask_SA[511:1024],color='gray',alpha=0.2,linewidth=2)
ax[1,0].set_xlim(0,30)
ax[1,0].set_ylim(0,1.1)
ax[1,0].set_xlabel('Baseline (m)')
ax[1,0].set_title('$uv$ coverage and filtering',fontsize=fs)


im = ax[1,1].imshow(abs(SA_feather+ST_feather),origin='lower',vmin=0,vmax=30,cmap='gray')
ax[1,1].set_title('Combined reconstruction',fontsize=fs)
ax2_divider = make_axes_locatable(ax[1,1])
cax2 = ax2_divider.append_axes("right", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="vertical",label='K')

frac_diff = (abs(SA_feather+ST_feather)-image_full)/image_full
im = ax[1,2].imshow(frac_diff,origin='lower',vmin=-0.05,vmax=0.05,cmap='bwr')
ax[1,2].set_title('True - combined',fontsize=fs)
ax2_divider = make_axes_locatable(ax[1,2])
cax2 = ax2_divider.append_axes("right", size="5%", pad="2%")
cb2 = fig.colorbar(im, cax=cax2, orientation="vertical",label='Fractional residuals')

print(np.min(frac_diff),np.max(frac_diff))

for i in range(0,3):
    for j in range(0,2):
        if j == 0:
            ax[j,i].xaxis.set_tick_params(labelbottom=False)
            ax[j,i].yaxis.set_tick_params(labelleft=False)
            ax[j,i].set_xticks([])
            ax[j,i].set_yticks([])
        else:
            if i != 0:
                ax[j,i].xaxis.set_tick_params(labelbottom=False)
                ax[j,i].yaxis.set_tick_params(labelleft=False)
                ax[j,i].set_xticks([])
                ax[j,i].set_yticks([])

plt.tight_layout()
plt.savefig('../CGPS_GMIMS_plots/filtering.png')

## Try different combinations of feathering parameters and optimize:

In [ ]:
dR = 0.5
R1_arr, R2_arr, mindiff, maxdiff = feathering_tests(ST,image_full,SA_deconv_FFT,dxy,pix0,dR)
find_optimized_feathering(R1_arr,R2_arr,dR,mindiff,maxdiff,SA_noise,ST_R)


# (2) Run different simulations and optimize feathering for each

In [ ]:
SA_noise_arr = [0, 500, 1000, 1500, 2000, 2500, 3000, 3500, 4000]
ST_R_arr = [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

dR = 0.5

for SA_noise in SA_noise_arr:
    for ST_R in ST_R_arr:
        
        print('--------------------------------------------------')
        print('Simulation for SA noise = '+str(SA_noise)+' and ST edge = '+str(ST_R))
        print('--------------------------------------------------')
        ST = ST_observe(image_full,image_ST_true,dxy,pix0,R2=ST_R,plots=False)
        SA_beam = get_beam(image_full,dxy,pix0,R=9,plots=False)
        SA_deconv_FFT = SA_deconvolve2(image_full, SA_beam, SA_noise,plots=False)

        R1_arr, R2_arr, mindiff, maxdiff = feathering_tests(ST,image_full,SA_deconv_FFT,dxy,pix0,dR)
        find_optimized_feathering(R1_arr,R2_arr,dR,mindiff,maxdiff,SA_noise,ST_R)
        
        print('')

In [ ]:
gap_image,gap_mask,uv_m = simple_gap(image_full,dxy,pix0)
plt.plot(uv_m,gap_mask[pix0,:])
plt.xlim(0,50)
plt.grid()

fig,ax = plt.subplots(1,3,figsize=(20,6))
ax[0].imshow(abs(gap_image),origin='lower',vmin=0,vmax=30)
ax[1].imshow(image_full,origin='lower',vmin=0,vmax=30)

frac_diff = (abs(gap_image)-image_full)/image_full

ax[2].imshow(frac_diff,origin='lower',vmin=-0.05,vmax=0.05)
print(np.min(frac_diff),np.max(frac_diff))